# ADP PM Notebook 3 — RAG Guardrails, RBAC and Observability

**Purpose:** Show why enterprise RAG must retrieve not only relevant data, but also authorized data.




## Classroom framing

A RAG answer can fail in two different ways:

1. **Relevance failure:** the wrong information is retrieved.
2. **Permission failure:** the right information is retrieved but the user is not allowed to see it.

PMs must design for both.


In [ ]:
# Uncomment only if needed.
# %pip install -q boto3 pandas numpy


In [ ]:
import os
import json
import time
import boto3
import numpy as np
import pandas as pd
from IPython.display import display

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
BEDROCK_LLM_MODEL_ID = "amazon.nova-lite-v1:0"
BEDROCK_EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v1"

bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
print("AWS Region:", AWS_REGION)


## 1. Small enterprise-style corpus

For speed, this notebook uses a compact in-notebook corpus. Replace this with S3 data if needed.


In [ ]:
policy_rows = [
    {
        "doc_type": "policy",
        "source": "HR-POL-008_Leave_Policy.docx",
        "text": "Employees are eligible for annual leave subject to manager approval and business continuity needs.",
        "required_level": 0,
    },
    {
        "doc_type": "policy",
        "source": "HR-POL-009_Remote_Work_Policy.docx",
        "text": "Remote work is allowed up to two days per week with manager approval and data security compliance.",
        "required_level": 0,
    },
]

salary_rows = [
    {
        "doc_type": "salary",
        "source": "salary_roster.csv",
        "employee_id": "EMP-1001",
        "text": "Employee ID EMP-1001, Name Asha Rao, Level L4, Role Product Analyst, Annual Salary INR 1800000.",
        "required_level": 6,
    },
    {
        "doc_type": "salary",
        "source": "salary_roster.csv",
        "employee_id": "EMP-1002",
        "text": "Employee ID EMP-1002, Name Kiran Shah, Level L6, Role Product Manager, Annual Salary INR 3400000.",
        "required_level": 6,
    },
]

corpus_df = pd.DataFrame(policy_rows + salary_rows)
corpus_df["chunk_id"] = [f"row_{i:03d}" for i in range(len(corpus_df))]
display(corpus_df)


## 2. Embed and retrieve

This is the same RAG idea, but now each retrieved row also has a permission requirement.


In [ ]:
def embed_text(text: str) -> list[float]:
    response = bedrock_runtime.invoke_model(
        modelId=BEDROCK_EMBEDDING_MODEL_ID,
        body=json.dumps({"inputText": text}),
        accept="application/json",
        contentType="application/json",
    )
    return json.loads(response["body"].read())["embedding"]


def cosine_similarity(a, b) -> float:
    a = np.array(a)
    b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

corpus_df["embedding"] = corpus_df["text"].apply(embed_text)
print("Corpus embedded")


In [ ]:
def retrieve_with_scores(query: str, top_k: int = 4) -> pd.DataFrame:
    q_embedding = embed_text(query)
    scored = corpus_df.copy()
    scored["score"] = scored["embedding"].apply(lambda e: cosine_similarity(q_embedding, e))
    return scored.sort_values("score", ascending=False).head(top_k)

retrieve_with_scores("What is Asha's salary?")[["source", "doc_type", "required_level", "score", "text"]]


## 3. Apply RBAC before answer generation

The model should only receive context that the requester is authorized to see.


In [ ]:
def parse_level(level: str) -> int:
    return int(str(level).upper().replace("L", ""))

requester_profile = {
    "employee_id": "EMP-2001",
    "name": "Demo Product Manager",
    "level": "L5",
}


def apply_rbac(retrieved_df: pd.DataFrame, requester_level: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    requester_level_num = parse_level(requester_level)
    allowed = retrieved_df[retrieved_df["required_level"] <= requester_level_num].copy()
    blocked = retrieved_df[retrieved_df["required_level"] > requester_level_num].copy()
    return allowed, blocked

query = "What is Asha's salary?"
retrieved = retrieve_with_scores(query)
allowed, blocked = apply_rbac(retrieved, requester_profile["level"])

print("Allowed rows:")
display(allowed[["source", "doc_type", "required_level", "text"]])
print("Blocked rows:")
display(blocked[["source", "doc_type", "required_level", "text"]])


## 4. Generate answer using authorized context only


In [ ]:
def call_llm(prompt: str) -> dict:
    start = time.time()
    response = bedrock_runtime.converse(
        modelId=BEDROCK_LLM_MODEL_ID,
        system=[{"text": "You are an enterprise assistant. Answer only from authorized context. If authorized evidence is not available, say access is restricted or evidence is missing."}],
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"temperature": 0.1, "topP": 0.9, "maxTokens": 260},
    )
    return {
        "answer": response["output"]["message"]["content"][0]["text"],
        "usage": response.get("usage", {}),
        "latency_ms": round((time.time() - start) * 1000, 2),
    }


def guarded_rag_answer(query: str, requester_profile: dict) -> dict:
    retrieved = retrieve_with_scores(query)
    allowed, blocked = apply_rbac(retrieved, requester_profile["level"])
    context = "\n\n".join([f"Source: {r.source}\n{r.text}" for r in allowed.itertuples()])
    prompt = f"""
Requester: {requester_profile}
Question: {query}

Authorized context:
{context if context else 'No authorized context available.'}

Answer clearly. Do not reveal blocked information.
"""
    llm_result = call_llm(prompt)
    return {
        "query": query,
        "requester_level": requester_profile["level"],
        "retrieved_count": len(retrieved),
        "allowed_count": len(allowed),
        "blocked_count": len(blocked),
        "blocked_sources": " | ".join(blocked["source"].astype(str).unique()),
        "answer": llm_result["answer"],
        "latency_ms": llm_result["latency_ms"],
    }

audit_row = guarded_rag_answer("What is Asha's salary?", requester_profile)
print(audit_row["answer"])


## 5. Final audit summary

This is what PMs should ask engineering teams to expose during evaluation.


In [ ]:
audit_df = pd.DataFrame([
    guarded_rag_answer("What is the remote work policy?", requester_profile),
    guarded_rag_answer("What is Asha's salary?", requester_profile),
])
display(audit_df)


## PM reflection checkpoint

- Was the answer blocked because retrieval failed or because access control worked?
- What should be visible to an auditor?
- What should be visible to the end user?
- How should this be written in product requirements?
